In [1]:
import torch
import imageio
import os
import csv 
import numpy as np 



In [2]:
# Loading an image file 

 
img_arr = imageio.imread("data/image-dog/bobby.jpg") # image is converted to numpy array. 
img_arr.shape # output are width, height and channels

(720, 1280, 3)

Pytorch modules dealing with image data require tensor to be laid out as C x H x W: channels, height and width. 

In [3]:
# given the layout H x W x C, we need a proper layout (C x H x W) by having channel 2 first and then channel 0 and 1
img = torch.from_numpy(img_arr)
out = img.permute(2, 0, 1) #C=2, H=0, W=1 


In [4]:
"""
So far, we have described a single image. Following the same strategy we’ve used
for earlier data types, to create a dataset of multiple images to use as an input for our
neural networks, we store the images in a batch along the first dimension to obtain an
N × C × H × W tensor.

As a slightly more efficient alternative to using stack to build up the tensor, we can pre-
allocate a tensor of appropriate size and fill it with images loaded from a directory,
"""
batch_size = 3 
batch = torch.zeros(batch_size, 3, 256, 256, dtype=torch.uint8) #batch will consiit of the RBG images 256 pixels in height and 256 pixels of width. 

In [5]:
#  loading PNG images from an input directory and store them in a tensor 

data_dir = "data/image-cats"
filenames = [name for name in os.listdir(data_dir) if os.path.splitext(name)[-1] == ".png"]
for i, filename in enumerate(filenames):
    img_arr = imageio.imread(os.path.join(data_dir, filename))
    img_t = torch.from_numpy(img_arr) 
    img_t = img_t.permute(2, 0, 1) 
    img_t = img_t[:3]
    batch[i] = img_t


In [6]:
# Normalizing the data.  
batch = batch.float()  # casting a tenosr to floating point and normalize the value of the pixel 
batch /= 255.0  # Dividing the pixel value by 255 (max. representable number in 8-bit unsigned)

In [7]:
# Normalizing the data by computing the mean and std of the input and scaling it  
n_channels = batch.shape[1] 
for c in range(n_channels): 
    mean = torch.mean(batch[:, c])
    std = torch.std(batch[:, c])
    batch[:, c] = (batch[:, c] - mean) / std

In [8]:
# 3D images: Volumetric data 
"""
Some images such as CT scans (medical images) are in 3D. 

CTs have only a single intensity channel, similar to a grayscale image. This means
that often, the channel dimension is left out in native data formats; so, similar to the
last section, the raw data typically has three dimensions. By stacking individual 2D
slices into a 3D tensor, we can build volumetric data representing the 3D anatomy of a
subject. 

In 3D image, we have extra dimension, i.e. depth, after the channel dimension, leading to a 5D tensor of shape  N x C x D x H x W. 
N = batch 
C = channel 
D = depth 
H = Height 
W = Width 
"""


# loading a sample CT scan. 

dir_path = "data/volumetric-dicom/2-LUNG 3.0  B70f-04083"
vol_arr = imageio.volread(dir_path, "DICOM")
vol_arr.shape

Reading DICOM (examining files):1/99 files (1.0%18/99 files (18.2%55/99 files (55.6%86/99 files (86.9%99/99 files (100.0%)
  Found 1 correct series.
Reading DICOM (loading data):13/99  (13.129/99  (29.345/99  (45.548/99  (48.551/99  (51.552/99  (52.553/99  (53.555/99  (55.663/99  (63.671/99  (71.778/99  (78.887/99  (87.994/99  (94.996/99  (97.099/99  (100.0%)


(99, 512, 512)

In [9]:
# since the data above does not have channel information, we make room for one using unsqueeze 
vol = torch.from_numpy(vol_arr).float() 
vol = torch.unsqueeze(vol, 0) 
vol.shape

torch.Size([1, 99, 512, 512])

In [10]:
# Representing Tabular Data
"""

"""

# loading tabular data using csv 
wine_path = "data/tabular-wine/winequality-white.csv"
wineeq_numpy = np.loadtxt(wine_path, dtype=np.float32, delimiter=";", skiprows=1)
wineeq_numpy

array([[ 7.  ,  0.27,  0.36, ...,  0.45,  8.8 ,  6.  ],
       [ 6.3 ,  0.3 ,  0.34, ...,  0.49,  9.5 ,  6.  ],
       [ 8.1 ,  0.28,  0.4 , ...,  0.44, 10.1 ,  6.  ],
       ...,
       [ 6.5 ,  0.24,  0.19, ...,  0.46,  9.4 ,  6.  ],
       [ 5.5 ,  0.29,  0.3 , ...,  0.38, 12.8 ,  7.  ],
       [ 6.  ,  0.21,  0.38, ...,  0.32, 11.8 ,  6.  ]], dtype=float32)

In [11]:
# checking if all data  has been read and also getting list of columns 
col_list = next(csv.reader(open(wine_path), delimiter=";"))
wineeq_numpy.shape, col_list

((4898, 12),
 ['fixed acidity',
  'volatile acidity',
  'citric acid',
  'residual sugar',
  'chlorides',
  'free sulfur dioxide',
  'total sulfur dioxide',
  'density',
  'pH',
  'sulphates',
  'alcohol',
  'quality'])

In [12]:
# convering numpy array to pytorch tensor 
wineeq = torch.from_numpy(wineeq_numpy) 
wineeq.shape, wineeq.dtype

(torch.Size([4898, 12]), torch.float32)

In [13]:
# representing scores 
data = wineeq[:, :-1] # select all rows and all columns execept the last 
target = wineeq[:, -1] # select all rows and the last column

data, data.shape, target, target.shape

(tensor([[ 7.0000,  0.2700,  0.3600,  ...,  3.0000,  0.4500,  8.8000],
         [ 6.3000,  0.3000,  0.3400,  ...,  3.3000,  0.4900,  9.5000],
         [ 8.1000,  0.2800,  0.4000,  ...,  3.2600,  0.4400, 10.1000],
         ...,
         [ 6.5000,  0.2400,  0.1900,  ...,  2.9900,  0.4600,  9.4000],
         [ 5.5000,  0.2900,  0.3000,  ...,  3.3400,  0.3800, 12.8000],
         [ 6.0000,  0.2100,  0.3800,  ...,  3.2600,  0.3200, 11.8000]]),
 torch.Size([4898, 11]),
 tensor([6., 6., 6.,  ..., 6., 7., 6.]),
 torch.Size([4898]))

In [14]:
# to transform the target tensor in a tensor of labels, thenw we can do as follows 
target = wineeq[:, -1].long() 
target

tensor([6, 6, 6,  ..., 6, 7, 6])

In [15]:
# One-hot encoding: This can be achieved using the scatter_ method.  
target_onehot = torch.zeros(target.shape[0], 10) 
target_onehot.scatter_(1, target.unsqueeze(1), 1.0) #args of scatter_; dimension, column tensor, a tensor 


tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [16]:
# obtaining the mean and variance for the data tensor 
data_mean = torch.mean(data, dim=0)
data_var = torch.var(data, dim=0)
print(data_mean, data_mean.shape)
print(data_var, data_var.shape)

tensor([6.8548e+00, 2.7824e-01, 3.3419e-01, 6.3914e+00, 4.5772e-02, 3.5308e+01,
        1.3836e+02, 9.9403e-01, 3.1883e+00, 4.8985e-01, 1.0514e+01]) torch.Size([11])
tensor([7.1211e-01, 1.0160e-02, 1.4646e-02, 2.5726e+01, 4.7733e-04, 2.8924e+02,
        1.8061e+03, 8.9455e-06, 2.2801e-02, 1.3025e-02, 1.5144e+00]) torch.Size([11])


In [17]:
# normalizing the data 
data_normalized = (data - data_mean) / torch.sqrt(data_var)
data_normalized

tensor([[ 1.7209e-01, -8.1764e-02,  2.1325e-01,  ..., -1.2468e+00,
         -3.4914e-01, -1.3930e+00],
        [-6.5743e-01,  2.1587e-01,  4.7991e-02,  ...,  7.3992e-01,
          1.3467e-03, -8.2418e-01],
        [ 1.4756e+00,  1.7448e-02,  5.4378e-01,  ...,  4.7502e-01,
         -4.3677e-01, -3.3662e-01],
        ...,
        [-4.2042e-01, -3.7940e-01, -1.1915e+00,  ..., -1.3131e+00,
         -2.6152e-01, -9.0544e-01],
        [-1.6054e+00,  1.1666e-01, -2.8253e-01,  ...,  1.0048e+00,
         -9.6250e-01,  1.8574e+00],
        [-1.0129e+00, -6.7703e-01,  3.7852e-01,  ...,  4.7502e-01,
         -1.4882e+00,  1.0448e+00]])

In [18]:
# Finding thresholds 

bad_indexes = target <= 3 #checking for scores less or equal to 3  
print(bad_indexes.shape, bad_indexes.dtype, bad_indexes.sum(),"\n")

bad_data = data[bad_indexes] # using advanced indexing to index the data tensor . 
print(bad_data.shape)

torch.Size([4898]) torch.bool tensor(20) 

torch.Size([20, 11])


In [19]:
# categorizing wine
bad_data = data[target <= 3]
mid_data = data[(target > 3) & (target < 7)]
good_data = data[target >= 7]

bad_mean = torch.mean(bad_data, dim=0)
mid_mean = torch.mean(mid_data, dim=0)
good_mean = torch.mean(good_data, dim=0)

for i, args in enumerate(zip(col_list, bad_mean, mid_mean, good_mean)):
    print('{:2} {:20} {:6.2f} {:6.2f} {:6.2f}'.format(i, *args))

0 fixed acidity          7.60   6.89   6.73
 1 volatile acidity       0.33   0.28   0.27
 2 citric acid            0.34   0.34   0.33
 3 residual sugar         6.39   6.71   5.26
 4 chlorides              0.05   0.05   0.04
 5 free sulfur dioxide   53.33  35.42  34.55
 6 total sulfur dioxide 170.60 141.83 125.25
 7 density                0.99   0.99   0.99
 8 pH                     3.19   3.18   3.22
 9 sulphates              0.47   0.49   0.50
10 alcohol               10.34  10.26  11.42


In [20]:
total_sulfur_threshold = 141.83 # midpoint value for sulphur 
total_sulfur_data = data[:, 6]  
predicted_indexes = torch.lt(total_sulfur_data, total_sulfur_threshold) # comparing the midpoint values to suphur threshold in each wine. 

print(predicted_indexes.shape, predicted_indexes.dtype, predicted_indexes.sum())

torch.Size([4898]) torch.bool tensor(2727)


In [21]:
# index of acutally good wines 
actual_indexes = target > 5 
print(actual_indexes.shape, actual_indexes.dtype, actual_indexes.sum())

torch.Size([4898]) torch.bool tensor(3258)


In [22]:
n_matches = torch.sum(actual_indexes & predicted_indexes).item()
n_predicted = torch.sum(predicted_indexes).item()  
n_actual = torch.sum(actual_indexes).item() 

print(n_matches, n_matches/n_predicted, n_matches / n_actual)

2018 0.74000733406674 0.6193984039287906


In [23]:
# Working with time series.  

"""
Goal is to take transform both Washington DC bikeshare system along with whether information from 2D to 3D.

The neural network model will need to see a number of sequences of values for
each different quantity, such as ride count, time of day, temperature, and weather con-
ditions: N parallel sequences of size C. C stands for channel, in neural network par-
lance, and is the same as column for 1D data like we have here. The N dimension
represents the time axis, here one entry per hour.

"""
bikes_numpy = np.loadtxt("data/bike-sharing-dataset/hour-fixed.csv", dtype=np.float32, delimiter=",", skiprows=1, converters={1: lambda x: float(x[8:10])}) 
bikes = torch.from_numpy(bikes_numpy) 
bikes

tensor([[1.0000e+00, 1.0000e+00, 1.0000e+00,  ..., 3.0000e+00, 1.3000e+01,
         1.6000e+01],
        [2.0000e+00, 1.0000e+00, 1.0000e+00,  ..., 8.0000e+00, 3.2000e+01,
         4.0000e+01],
        [3.0000e+00, 1.0000e+00, 1.0000e+00,  ..., 5.0000e+00, 2.7000e+01,
         3.2000e+01],
        ...,
        [1.7377e+04, 3.1000e+01, 1.0000e+00,  ..., 7.0000e+00, 8.3000e+01,
         9.0000e+01],
        [1.7378e+04, 3.1000e+01, 1.0000e+00,  ..., 1.3000e+01, 4.8000e+01,
         6.1000e+01],
        [1.7379e+04, 3.1000e+01, 1.0000e+00,  ..., 1.2000e+01, 3.7000e+01,
         4.9000e+01]])

In [24]:
# shaping the data by time period 

"""
Time series dataset would be of tensor of dimension 3 and shape N x C x L. 
Channel = 17 (ie. variables from the dataset)
L = 24: 1 per hour of the day (ie. the day is made of 24 hours). We could also use 7*24 = 168 hours which represent a week instead of using days. 

"""
print(bikes.shape, bikes.stride()) # output show 17,520 hours and 17 columns. 

# reshaping the data to have 3 axes - day, hour, then 17 columns 

daily_bikes = bikes.view(-1, 24, bikes.shape[1])
daily_bikes = daily_bikes.transpose(1,2) # transposing to get N x C x L odering. 
print(daily_bikes.shape, daily_bikes.stride())


torch.Size([17520, 17]) (17, 1)
torch.Size([730, 17, 24]) (408, 1, 17)


In [25]:
# Ready for traing. 

"""
Weather situation variable is ordinal. Thus we need to apply one-hot encoding and concatenate the columns with the dataset 
"""
first_day = bikes[:24].long() #extracting first 24 hours from the bike data and conveting it to integ
weather_onehot = torch.zeros(first_day.shape[0], 4) # initializing a zero filled matrix equal to number of hours in a day and number of levels (ie. 4)
weather_onehot.scatter_(dim=1, index=first_day[:,9].unsqueeze(1).long() - 1, value=1.0) # index is decreased by 1 because weather situatioon ranges form 1 to 4, while indicies are 0 based. 
torch.cat((bikes[:24], weather_onehot), 1)[:1] # concating matric to original dataset using the cat function. 



tensor([[ 1.0000,  1.0000,  1.0000,  0.0000,  1.0000,  0.0000,  0.0000,  6.0000,
          0.0000,  1.0000,  0.2400,  0.2879,  0.8100,  0.0000,  3.0000, 13.0000,
         16.0000,  1.0000,  0.0000,  0.0000,  0.0000]])

In [26]:
# applying the same principle above to daily weather data 
daily_weather_onehot = torch.zeros(daily_bikes.shape[0], 4, daily_bikes.shape[2])
print(daily_weather_onehot.shape)

daily_weather_onehot.scatter_(dim=1, index= daily_bikes[:, 9, :].long().unsqueeze(1) - 1, value=1.0)
print(daily_weather_onehot.shape)

daily_bikes = torch.cat((daily_bikes, daily_weather_onehot), dim=1) 

torch.Size([730, 4, 24])
torch.Size([730, 4, 24])


In [27]:
# also we can treat our weather situation variable as a special values of continuous varaible, which runs from 0.0 to 1.0. This can be 
daily_bikes[:, 9, :] = (daily_bikes[:, 9, :] - 1.0)/3.0

# also rescaling temperature (column 10 in our dataset)
temp = daily_bikes[:, 10, :] 
temp_min = torch.min(temp)
temp_max = torch.max(temp) 
daily_bikes[:, 10, :] = ((daily_bikes[:, 10, :] - temp_min)/(temp_max - temp_min)) # scales from 0.0 to 0.1 

"""
#using the above method or the method below for rescaling. 
temp_mean = torch.mean(temp) 
temp_std = torch.std(temp)
daily_bikes[:, 10, :] = ((daily_bikes[:, 10, :] - temp_mean)/temp_std) 
"""

'\n#using the above method or the method below for rescaling. \ntemp_mean = torch.mean(temp) \ntemp_std = torch.std(temp)\ndaily_bikes[:, 10, :] = ((daily_bikes[:, 10, :] - temp_mean)/temp_std) \n'

In [28]:
# Representing Text 
# conveerting text to numbers 
with open("data/jane-austen/1342-0.txt", encoding="utf8") as f:
    text = f.read()

# one-hot-encoding characters
lines = text.split("\n")
line = lines[200] 
print(line)




# performing one-hot encoding for characters. 
letter_t = torch.zeros(len(line), 128) # Initializing a zero matrix. 128 hardcoded due to the limit of ASCII 
print(letter_t.shape)

for i, letter in enumerate(line.lower().strip()):
    letter_index = ord(letter) if ord(letter) < 128 else 0 
    letter_t[i][letter_index] = 1


“Impossible, Mr. Bennet, impossible, when I am not acquainted with him
torch.Size([70, 128])


In [29]:
# one-hot encoding words 
"""
More efficeint way to repsent text is using embeddings.  
Below we create a function that clean up text by removing punctuation and returning words in lowercase.  
"""

def clean_words(input_str):
    punctuation = '.,;:"!?”“_-'
    word_list = input_str.lower().replace("\n", " ").split() 
    word_list = [word.strip(punctuation) for word in word_list]
    return word_list

words_in_line = clean_words(line)
print(line,"\n", words_in_line, len(words_in_line),"\n")


# mapping of word(text) to index in our encoding 
word_list = sorted(set(clean_words(text))) 
word2index_dict = {word:i for (i, word) in enumerate(word_list)} #dict. with words as keys and index (int) as values.

print(len(word2index_dict), word2index_dict['impossible'])


“Impossible, Mr. Bennet, impossible, when I am not acquainted with him 
 ['impossible', 'mr', 'bennet', 'impossible', 'when', 'i', 'am', 'not', 'acquainted', 'with', 'him'] 11 

7261 3394


In [32]:
# performing one hot-conding on words  
word_t = torch.zeros(len(words_in_line), len(word2index_dict))
for i, word in enumerate(words_in_line):
    word_index = word2index_dict[word]
    word_t[i][word_index] = 1 
    print("{:2} {:4} {}".format(i, word_index, word))

print(word_t.shape)


0 3394 impossible
 1 4305 mr
 2  813 bennet
 3 3394 impossible
 4 7078 when
 5 3315 i
 6  415 am
 7 4436 not
 8  239 acquainted
 9 7148 with
10 3215 him
torch.Size([11, 7261])


In [49]:
# Text embeddings 
"""
One-hot encoding is a very useful technique for representing categorical data in ten-
sors. However, as we have anticipated, one-hot encoding starts to break down when
the number of items to encode is effectively unbound, as with words in a corpus.

How can we compress our encoding down to a more manageable size and put a
cap on the size growth? Well, instead of vectors of many zeros and a single one, we can
use vectors of floating-point numbers. A vector of, say, 100 floating-point numbers can
indeed represent a large number of words. The trick is to find an effective way to map
individual words into this 100-dimensional space in a way that facilitates downstream
learning. This is called an embedding.

One interesting aspect of the resulting embeddings is that similar words end up not
only clustered together, but also having consistent spatial relationships with other
words

"""

# Text embeddings as a blueprint 


<generator object <genexpr> at 0x7f77886e33d0>
